# 🎬 VoxStudio Server — Google Colab

Chạy VoxStudio server trên Colab **miễn phí GPU T4**

> ⚠️ Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy!

In [ ]:
#@title 1️⃣ Kiểm tra GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ Không có GPU! Vào Runtime → Change runtime type → T4 GPU")

In [ ]:
#@title 2️⃣ Lấy project code
#@markdown ### Chọn 1 trong 2 cách:
#@markdown - **Cách 1 (GitHub)**: Clone từ repo — không cần upload lại mỗi lần sửa
#@markdown - **Cách 2 (Drive)**: Upload thư mục lên Google Drive

CODE_SOURCE = "drive"  #@param ["github", "drive"]
GITHUB_REPO = ""  #@param {type:"string"}
#@markdown ↑ Nếu dùng GitHub: nhập URL repo (vd: `https://github.com/user/VoxStudio`)

import os, shutil

if CODE_SOURCE == "github" and GITHUB_REPO:
    # ── GitHub: clone hoặc pull mới nhất ──
    repo_url = GITHUB_REPO.strip().rstrip("/")
    if not repo_url.endswith(".git"):
        repo_url += ".git"

    if os.path.exists("/content/VoxStudio/.git"):
        print("📥 Pulling latest code...")
        !cd /content/VoxStudio && git pull
    else:
        print("📥 Cloning repo...")
        !git clone {repo_url} /content/VoxStudio

    # Copy server
    if os.path.exists("/content/server"):
        shutil.rmtree("/content/server", ignore_errors=True)
    !cp -r /content/VoxStudio/server /content/server

    # OmniVoice — vẫn cần từ Drive hoặc clone riêng
    if not os.path.exists("/content/OmniVoice-master"):
        print("\n⚠️ Cần OmniVoice-master! Upload lên Drive hoặc clone riêng")
        from google.colab import drive
        drive.mount('/content/drive')
        !cp -r "/content/drive/MyDrive/OmniVoice-master" /content/OmniVoice-master

else:
    # ── Google Drive: upload thủ công ──
    #@markdown Cấu trúc Drive:
    #@markdown ```
    #@markdown MyDrive/
    #@markdown   OmniVoice-master/     ← TTS model source
    #@markdown   VoxStudio/
    #@markdown     server/             ← Server code
    #@markdown ```
    from google.colab import drive
    drive.mount('/content/drive')

    # Copy server code (xóa cũ trước để đảm bảo code mới nhất)
    if os.path.exists("/content/server"):
        shutil.rmtree("/content/server")
    !cp -r "/content/drive/MyDrive/VoxStudio/server" /content/server

    # Copy OmniVoice source
    if not os.path.exists("/content/OmniVoice-master"):
        !cp -r "/content/drive/MyDrive/OmniVoice-master" /content/OmniVoice-master

print("\n=== Server ===")
!ls /content/server/
print("\n=== OmniVoice ===")
!ls /content/OmniVoice-master/
print("\n✅ Code ready!")

In [ ]:
#@title 3️⃣ Cài đặt dependencies
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and 'ERROR' in (r.stderr or ''):
        print(f"⚠️ {cmd}: {r.stderr.strip().split(chr(10))[-1]}")
    return r.returncode == 0

print("📦 Installing core packages...")
run("pip install -q fastapi 'uvicorn[standard]' python-multipart soundfile pyngrok")

print("📦 Installing translation...")
run("pip install -q deep-translator google-generativeai")

print("📦 Installing AI models...")
run("pip install -q transformers accelerate")

print("📦 Installing Faster-Whisper + scipy...")
run("pip install -q faster-whisper scipy")

print("📦 Installing OmniVoice TTS from source...")
result = run("pip install -q -e /content/OmniVoice-master")
if result:
    print("  ✅ OmniVoice installed!")
else:
    print("  ❌ OmniVoice install failed — check /content/OmniVoice-master exists")

print("📦 Installing Demucs...")
run("pip install -q demucs")

print("📦 Installing ffmpeg...")
run("apt-get install -y -qq ffmpeg > /dev/null 2>&1")

print("📦 Installing remaining from requirements.txt...")
run("cd /content/server && pip install -q -r requirements.txt")

# Verify OmniVoice
print("\n=== Verify ===")
try:
    import omnivoice
    print(f"✅ omnivoice {omnivoice.__version__}")
except ImportError as e:
    print(f"❌ omnivoice: {e}")

try:
    import torch
    print(f"✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})")
except: pass

try:
    import transformers
    print(f"✅ transformers {transformers.__version__}")
except: pass

try:
    from faster_whisper import WhisperModel
    print("✅ faster-whisper installed")
except ImportError as e:
    print(f"❌ faster-whisper: {e}")

try:
    import scipy
    print(f"✅ scipy {scipy.__version__}")
except ImportError as e:
    print(f"❌ scipy: {e}")

print("\n✅ All dependencies installed!")

In [ ]:
#@title 4️⃣ Setup ngrok
#@markdown Lấy token **miễn phí** tại: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = ""  #@param {type:"string"}

if not NGROK_TOKEN:
    print("❌ Cần nhập NGROK_TOKEN!")
    print("👉 Đăng ký: https://dashboard.ngrok.com/signup")
    print("👉 Copy token: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok configured!")

In [ ]:
#@title 5️⃣ Cấu hình
GEMINI_API_KEY = ""  #@param {type:"string"}

import os
os.environ["DEVICE"] = "cuda"
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("✅ Gemini API key set")
print("✅ Device: cuda")

In [ ]:
#@title 6️⃣ 🚀 Khởi động Server

import subprocess, time, requests, re, sys, os
from pyngrok import ngrok

# Kill cũ
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
try: ngrok.kill()
except: pass
time.sleep(1)

# Start server
os.chdir("/content/server")
log_file = open("/content/server.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file, stderr=subprocess.STDOUT,
    cwd="/content/server",
)
print(f"⏳ Server PID: {server.pid}")
print("⏳ Loading Whisper model (~30-60s on first run)...")
print()

# Wait (max 5 min)
ready = False
for i in range(150):
    time.sleep(2)
    
    if server.poll() is not None:
        print(f"\n❌ Server crashed! Exit code: {server.returncode}")
        print("\n=== ERROR LOG (last 40 lines) ===")
        with open("/content/server.log") as f:
            for line in f.readlines()[-40:]:
                print(line, end="")
        break
    
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        data = r.json()
        if data.get("status") == "ok":
            print(f"\n✅ Server ready! ({(i+1)*2}s) — Device: {data.get('device')}")
            ready = True
            break
        else:
            print(f"  Loading models... ({(i+1)*2}s)")
    except requests.ConnectionError:
        if i % 5 == 0:
            # Show latest log line for progress
            try:
                with open("/content/server.log") as f:
                    lines = f.readlines()
                    last = lines[-1].strip() if lines else ""
                    print(f"  [{(i+1)*2}s] {last[:80]}")
            except:
                print(f"  ⏳ Starting... ({(i+1)*2}s)")
    except Exception as e:
        if i % 10 == 0:
            print(f"  Waiting... ({(i+1)*2}s)")

if not ready and server.poll() is None:
    print("\n⚠️ Chưa ready nhưng server vẫn đang chạy (có thể đang download model)")
    print("   Chạy lại cell này sau 1-2 phút, hoặc xem cell 📊 Logs")

# Ngrok tunnel
if server.poll() is None:
    tunnel = ngrok.connect(8000, "http")
    url = str(tunnel)
    match = re.search(r'https://[^"\s]+', url)
    public_url = match.group(0) if match else url
    
    print(f"\n{'='*60}")
    print(f"🌐 SERVER URL: {public_url}")
    print(f"{'='*60}")
    print(f"\n📋 Trên máy local chạy:")
    print(f"   cd desktop")
    print(f"   VITE_API_URL={public_url} npm run dev")
    print(f"\n🧪 Test: {public_url}/health")

In [ ]:
#@title 📊 Xem server logs
!echo "=== Process ==="
!ps aux | grep uvicorn | grep -v grep || echo "❌ Server not running"
!echo ""
!echo "=== Last 50 lines ==="
!tail -50 /content/server.log 2>/dev/null || echo "No log file"

In [ ]:
#@title 🧪 Test API
import requests
for name, url in [
    ("Health", "http://localhost:8000/health"),
    ("Voices", "http://localhost:8000/api/v1/voices"),
    ("Gemini", "http://localhost:8000/api/v1/dubbing/gemini-status"),
]:
    try:
        r = requests.get(url, timeout=5)
        print(f"{name}: {r.json()}")
    except Exception as e:
        print(f"{name}: ❌ {e}")

In [ ]:
#@title ⏹️ Dừng server
from pyngrok import ngrok
try: ngrok.kill()
except: pass
!pkill -f uvicorn
print("✅ Server stopped.")